# Import Libraries & Intsalled Required Packages

Install Packages

In [ ]:
!pip install monai
!pip install careamics

In [ ]:
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install causal-conv1d==1.4.0 && pip install mamba-ssm==2.2.2 --no-build-isolation

Clone MambaIRv2 repo

In [ ]:
!git clone https://github.com/csguoh/MambaIR.git

Import all libraries needed

In [ ]:
import torchvision
import sys
import os
import random
import numpy as np
import cv2
import albumentations as A
import numpy as np
import tifffile as tiff
from monai.transforms import Compose, MapTransform, RandomizableTransform
from monai.data import Dataset, DataLoader, CacheDataset
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import re
from pathlib import Path
from google.colab import drive
import matplotlib.pyplot as plt
import copy
import gc

try:
    from careamics.metrics import SampleSIPSNR
except ImportError:
    SampleSIPSNR = None

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torch.optim as optim

from tqdm.auto import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold

sys.path.append('/content/MambaIR')
from basicsr.archs.mambairv2_arch import MambaIRv2

Import google drive to get the dataset and save the checkpoints during training

In [ ]:
drive.mount('/content/drive')

# Global Variables

Choose one of: 'Nuclei', 'FMD', 'Planaria' or 'Tribolium'

In [ ]:
dataset = 'Nuclei'

## Extraction: only for 3d if needed

For the 3D datasets an extraction to 2D is needed

In [ ]:
if dataset == 'Planaria':
  DRIVE_BASE = '/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Planaria/'
  DRIVE_NOISY_SMALL = os.path.join(DRIVE_BASE, 'small_crops', 'noisy')
  DRIVE_NOISY_LARGE = os.path.join(DRIVE_BASE, 'large_crops', 'noisy')
  DRIVE_GT_SMALL = os.path.join(DRIVE_BASE, 'small_crops', 'gt')
  DRIVE_GT_LARGE = os.path.join(DRIVE_BASE, 'large_crops', 'gt')
  LOCAL_NOISY_SMALL = '/content/planaria_2d/noisy/small/'
  LOCAL_NOISY_LARGE = '/content/planaria_2d/noisy/large/'
  LOCAL_GT_SMALL = '/content/planaria_2d/gt/small/'
  LOCAL_GT_LARGE = '/content/planaria_2d/gt/large/'
elif dataset == 'Tribolium':
  DRIVE_BASE = '/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Tribolium/'
  DRIVE_NOISY_SMALL = os.path.join(DRIVE_BASE, 'small_crops', 'noisy')
  DRIVE_NOISY_LARGE = os.path.join(DRIVE_BASE, 'large_crops', 'noisy')
  DRIVE_GT_SMALL = os.path.join(DRIVE_BASE, 'small_crops', 'gt')
  DRIVE_GT_LARGE = os.path.join(DRIVE_BASE, 'large_crops', 'gt')
  LOCAL_NOISY = '/content/tribolium_2d/noisy/'
  LOCAL_GT = '/content/tribolium_2d/gt/'

In [ ]:
import os
import tifffile
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
import shutil

def process_single_volume(args):
  stack_path, output_2d_dir, base_name = args
  try:
    volume = tifffile.imread(stack_path)
    for z in range(volume.shape[0]):
      slice_img = volume[z]
      out_filename = f"{base_name}_z{z:03d}.tif"
      out_filepath = os.path.join(output_2d_dir, out_filename)
      tifffile.imwrite(out_filepath, slice_img)
    return True
  except Exception as e:
    return False

def unpack_3d_to_2d_parallel(input_3d_dir, output_2d_dir, max_workers=8):
  os.makedirs(output_2d_dir, exist_ok=True)
  stack_files = [f for f in os.listdir(input_3d_dir) if f.lower().endswith(('.tif', '.tiff'))]

  task_args = []
  for stack_name in stack_files:
    stack_path = os.path.join(input_3d_dir, stack_name)
    base_name = os.path.splitext(stack_name)[0]
    task_args.append((stack_path, output_2d_dir, base_name))


  with ThreadPoolExecutor(max_workers=max_workers) as executor:
    list(tqdm(executor.map(process_single_volume, task_args), total=len(task_args), desc="Unpacking to NVMe"))

In [ ]:
if dataset == 'Planaria':
  unpack_3d_to_2d_parallel(DRIVE_NOISY_SMALL, LOCAL_NOISY_SMALL, max_workers=2)
  unpack_3d_to_2d_parallel(DRIVE_NOISY_LARGE, LOCAL_NOISY_LARGE, max_workers=2)
  unpack_3d_to_2d_parallel(DRIVE_GT_SMALL, LOCAL_GT_SMALL, max_workers=2)
  unpack_3d_to_2d_parallel(DRIVE_GT_LARGE, LOCAL_GT_LARGE, max_workers=2)
  !cd /content && zip -r -q planaria_2d_dataset.zip planaria_2d
  !cp /content/planaria_2d_dataset.zip '/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Planaria/'
elif dataset == 'Tribolium':
  unpack_3d_to_2d_parallel(DRIVE_NOISY, LOCAL_NOISY, max_workers=2)
  unpack_3d_to_2d_parallel(DRIVE_GT, LOCAL_GT, max_workers=2)
  !cd /content && zip -r -q tribolium_2d_dataset.zip tribolium_2d
  !cp /content/tribolium_2d_dataset.zip '/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Tribolium/'

## Load Dataset

In [ ]:
match dataset:
  case 'Nuclei':
    NOISY_DATASET_SMALL='/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Nuclei/small_images/noisy/'
    CLEAN_DATASET_SMALL='/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Nuclei/small_images/gt/'
    NOISY_DATASET_LARGE='/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Nuclei/large_images/noisy/'
    CLEAN_DATASET_LARGE='/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Nuclei/large_images/gt/'
  case 'FMD':
    NOISY_DATASET='/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-FMD/noisy/'
    CLEAN_DATASET='/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-FMD/gt/'
  case 'Planaria':
    !cp "/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Planaria/planaria_2d_dataset.zip" "/content/"
    !unzip -q /content/planaria_2d_dataset.zip -d /content/
    NOISY_SMALL = '/content/planaria_2d/noisy/small/'
    CLEAN_SMALL = '/content/planaria_2d/gt/small/'
    NOISY_LARGE = '/content/planaria_2d/noisy/large/'
    CLEAN_LARGE = '/content/planaria_2d/gt/large/'
  case 'Tribolium':
    !cp "/content/drive/MyDrive/Master/AIDL/3 Semester/Thesis/Dataset/AI4Life-MDC25-Tribolium/tribolium_2d_dataset.zip" "/content/"
    !unzip -q /content/tribolium_2d_dataset.zip -d /content/
    NOISY_DATASET='/content/tribolium_2d/noisy/'
    CLEAN_DATASET='/content/tribolium_2d/gt/'

# ML Pipeline

## AlbumentationsDictTransform for 2D dataset

In [ ]:
def ensure_2d(img):
  img = np.asarray(img)
  if img.ndim == 2:
    return img

  if img.ndim == 3 and img.shape[0] == 1:
    return img[0]

  if img.ndim == 3 and img.shape[-1] == 1:
    return img[..., 0]

In [ ]:
class CustomTiffLoader(MapTransform):
  def __init__(self, keys=("noisy", "gt"), allow_missing_keys=False):
    super().__init__(keys, allow_missing_keys)

  def __call__(self, data):
    d = dict(data)
    for key in self.key_iterator(d):
      img = tiff.imread(d[key])
      d[key] = np.asarray(img)

    return d

In [ ]:
class IndependentPercentileScaler(MapTransform):
  def __init__(self, keys=("noisy", "gt"), lower=0.1, upper=99.9, eps=1e-8, clip=True):
    super().__init__(keys)
    self.lower = lower
    self.upper = upper
    self.eps = eps
    self.clip = clip

  def __call__(self, data):
    d = dict(data)
    for key in self.key_iterator(d):
      img = np.asarray(d[key]).astype(np.float32)

      v_min = np.percentile(img, self.lower)
      v_max = np.percentile(img, self.upper)

      scale = v_max - v_min
      if scale < self.eps:
        scale = self.eps

      img = (img - v_min) / scale

      if self.clip:
        img = np.clip(img, 0.0, 1.0)

      d[key] = img.astype(np.float32)

    return d

In [ ]:
class AlbumentationsDictTransform(RandomizableTransform, MapTransform):
  def __init__(self, keys=("noisy", "gt"), crop_size=256, min_patch_std=0.005, max_tries=10, allow_missing_keys=False):
    MapTransform.__init__(self, keys, allow_missing_keys)
    RandomizableTransform.__init__(self, prob=1.0)
    self.crop_size = crop_size
    self.min_patch_std = min_patch_std
    self.max_tries = max_tries
    self.albu_pipeline = A.Compose(
        [
            A.PadIfNeeded(min_height=crop_size, min_width=crop_size, border_mode=cv2.BORDER_REFLECT_101),
            A.RandomCrop(width=crop_size, height=crop_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.75),
        ],
        additional_targets={"gt": "image"},
    )

  def __call__(self, data):
    d = dict(data)
    noisy_np = ensure_2d(d["noisy"]).astype(np.float32)
    gt_np = ensure_2d(d["gt"]).astype(np.float32)
    transformed = None

    for _ in range(self.max_tries):
      candidate = self.albu_pipeline(image=noisy_np, gt=gt_np)

      if self.min_patch_std is None:
        transformed = candidate
        break

      if candidate["gt"].std() >= self.min_patch_std:
        transformed = candidate
        break

      transformed = candidate

    noisy_aug = transformed["image"].copy()
    gt_aug = transformed["gt"].copy()

    noisy_aug = np.ascontiguousarray(noisy_aug[None, ...]).astype(np.float32)
    gt_aug = np.ascontiguousarray(gt_aug[None, ...]).astype(np.float32)

    d["noisy"] = torch.from_numpy(noisy_aug).float()
    d["gt"] = torch.from_numpy(gt_aug).float()

    return d

In [ ]:
class ToTensor2D(MapTransform):
  def __init__(self, keys=("noisy", "gt"), allow_missing_keys=False):
    super().__init__(keys, allow_missing_keys)

  def __call__(self, data):
    d = dict(data)
    for key in self.key_iterator(d):
      img = ensure_2d(d[key]).astype(np.float32)
      img = np.ascontiguousarray(img[None, ...])
      d[key] = torch.from_numpy(img).float()

    return d

## AlbumentationsDictTransform for 3D dataset

In [ ]:
def ensure_2d(img):
  img = np.asarray(img)
  if img.ndim == 2:
    return img

  if img.ndim == 3 and img.shape[0] == 1:
    return img[0]

  if img.ndim == 3 and img.shape[-1] == 1:
    return img[..., 0]

In [ ]:
class CustomTiffLoader25D(MapTransform):
    def __init__(
        self,
        keys=("noisy", "gt"),
        expected_channels=5,
        allow_missing_keys=False,
    ):
        super().__init__(keys, allow_missing_keys)
        self.expected_channels = expected_channels

    def __call__(self, data):
        d = dict(data)

        if "noisy" in d:
            noisy_paths = d["noisy"]

            if not isinstance(noisy_paths, (list, tuple)):
                raise TypeError(
                    f"Expected d['noisy'] to be a list/tuple of paths, got {type(noisy_paths)}"
                )

            if len(noisy_paths) != self.expected_channels:
                raise ValueError(
                    f"Expected {self.expected_channels} noisy slices, got {len(noisy_paths)}"
                )

            noisy_slices = []

            for path in noisy_paths:
                img = tiff.imread(path)
                img = np.asarray(img, dtype=np.float32)

                if img.ndim != 2:
                    raise ValueError(f"Expected 2D noisy slice, got shape {img.shape} from {path}")

                noisy_slices.append(img)

            d["noisy"] = np.stack(noisy_slices, axis=0)  # C, H, W

        if "gt" in d:
            gt_img = tiff.imread(d["gt"])
            gt_img = np.asarray(gt_img, dtype=np.float32)

            if gt_img.ndim != 2:
                raise ValueError(f"Expected 2D GT slice, got shape {gt_img.shape} from {d['gt']}")

            d["gt"] = gt_img  # H, W

        return d

In [ ]:
class PairedPercentileScaler25D(MapTransform):
    """
    Normalizes both the 2.5D noisy input and the GT target
    using percentiles computed from the noisy input stack.

    noisy: shape (C, H, W), e.g. (5, H, W)
    gt:    shape (H, W)
    """

    def __init__(
        self,
        keys=("noisy", "gt"),
        reference_key="noisy",
        lower=0.1,
        upper=99.9,
        eps=1e-8,
        clip=True,
        save_stats=False,
        allow_missing_keys=False,
    ):
        super().__init__(keys, allow_missing_keys)

        self.reference_key = reference_key
        self.lower = lower
        self.upper = upper
        self.eps = eps
        self.clip = clip
        self.save_stats = save_stats

    def __call__(self, data):
        d = dict(data)

        ref = np.asarray(d[self.reference_key]).astype(np.float32)

        v_min = np.percentile(ref, self.lower)
        v_max = np.percentile(ref, self.upper)

        scale = v_max - v_min
        if scale < self.eps:
            scale = self.eps

        for key in self.key_iterator(d):
            img = np.asarray(d[key]).astype(np.float32)
            img = (img - v_min) / scale

            if self.clip:
                img = np.clip(img, 0.0, 1.0)

            d[key] = img.astype(np.float32)

        if self.save_stats:
            d["norm_v_min"] = np.float32(v_min)
            d["norm_scale"] = np.float32(scale)

        return d

In [ ]:
class AlbumentationsDictTransform25D(RandomizableTransform, MapTransform):
    """
    Applies identical spatial augmentations across all 5 input channels
    and aligns them with the single ground truth target.
    """
    def __init__(self, keys=("noisy", "gt"), crop_size=256, min_patch_std=0.005, max_tries=10, allow_missing_keys=False):
        MapTransform.__init__(self, keys, allow_missing_keys)
        RandomizableTransform.__init__(self, prob=1.0)
        self.crop_size = crop_size
        self.min_patch_std = min_patch_std
        self.max_tries = max_tries

        self.albu_pipeline = A.Compose(
            [
                A.PadIfNeeded(min_height=crop_size, min_width=crop_size, border_mode=cv2.BORDER_REFLECT_101),
                A.RandomCrop(width=crop_size, height=crop_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.75),
            ],
            additional_targets={"gt": "image"},
        )

    def __call__(self, data):
        d = dict(data)

        # 'noisy' shape coming in: (5, H, W) -> Convert to (H, W, 5) for Albumentations
        noisy_np = np.transpose(d["noisy"], (1, 2, 0)).astype(np.float32)
        gt_np = d["gt"].astype(np.float32)

        transformed = None

        for _ in range(self.max_tries):
            candidate = self.albu_pipeline(image=noisy_np, gt=gt_np)

            if self.min_patch_std is None:
                transformed = candidate
                break

            if candidate["gt"].std() >= self.min_patch_std:
                transformed = candidate
                break

            transformed = candidate

        # Extract outputs
        noisy_aug = transformed["image"].copy()  # Shape: (H, W, 5)
        gt_aug = transformed["gt"].copy()        # Shape: (H, W)

        # Transpose noisy back to PyTorch format: (Channels, Height, Width) -> (5, H, W)
        noisy_aug = np.transpose(noisy_aug, (2, 0, 1))

        # Add batch dimension proxy -> (1, Height, Width) for GT
        gt_aug = gt_aug[None, ...]

        d["noisy"] = torch.from_numpy(np.ascontiguousarray(noisy_aug)).float()
        d["gt"] = torch.from_numpy(np.ascontiguousarray(gt_aug)).float()

        return d

In [ ]:
class ToTensor25D(MapTransform):
    """Converts the 2.5D numpy arrays to PyTorch Tensors for validation/testing."""
    def __init__(self, keys=("noisy", "gt"), allow_missing_keys=False):
        super().__init__(keys, allow_missing_keys)

    def __call__(self, data):
        d = dict(data)
        if "noisy" in d:
            # CustomTiffLoader25D outputs (5, H, W). Just convert to tensor.
            noisy_np = d["noisy"].astype(np.float32)
            d["noisy"] = torch.from_numpy(np.ascontiguousarray(noisy_np)).float()

        if "gt" in d:
            # GT is (H, W). Add the channel proxy so it becomes (1, H, W).
            gt_np = d["gt"].astype(np.float32)
            gt_np = np.ascontiguousarray(gt_np[None, ...])
            d["gt"] = torch.from_numpy(gt_np).float()

        return d

## Read the dataset

### For 2D

In [ ]:
def natural_key(x):
  x = str(x)
  return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", x)]

def list_tiff_files(folder):
  folder = Path(folder)
  files = [
      p for p in folder.iterdir()
      if p.suffix.lower() in [".tif", ".tiff"]
  ]
  return sorted(files, key=natural_key)


def pair_key(path):
  return Path(path).stem


def build_pairs(noisy_dir, clean_dir, subset_name):
  noisy_files = list_tiff_files(noisy_dir)
  clean_files = list_tiff_files(clean_dir)
  print(f"{subset_name}: found {len(noisy_files)} noisy files")
  print(f"{subset_name}: found {len(clean_files)} clean files")

  noisy_map = {}
  clean_map = {}
  for p in noisy_files:
    k = pair_key(p)
    if k in noisy_map:
      raise ValueError(f"Duplicate noisy key '{k}' in {noisy_dir}")
    noisy_map[k] = p

  for p in clean_files:
    k = pair_key(p)
    if k in clean_map:
      raise ValueError(f"Duplicate clean key '{k}' in {clean_dir}")
    clean_map[k] = p

  noisy_keys = set(noisy_map.keys())
  clean_keys = set(clean_map.keys())

  missing_clean = sorted(noisy_keys - clean_keys, key=natural_key)
  missing_noisy = sorted(clean_keys - noisy_keys, key=natural_key)

  if missing_clean:
    print("No clean match for these noisy files:")
    print(missing_clean[:20])
    raise ValueError(f"{len(missing_clean)} noisy files have no clean pair.")

  if missing_noisy:
    print("No noisy match for these clean files:")
    print(missing_noisy[:20])
    raise ValueError(f"{len(missing_noisy)} clean files have no noisy pair.")

  pairs = []
  for k in sorted(noisy_keys, key=natural_key):
    pairs.append(
        {
            "noisy": str(noisy_map[k]),
            "gt": str(clean_map[k]),
            "id": f"{subset_name}_{k}",
            "subset": subset_name,
        }
    )

  return pairs

### For 3D


In [ ]:
from collections import defaultdict
import random


def split_25d_by_volume(
    data_dicts,
    train_ratio=0.8,
    val_ratio=0.1,
    seed=42,
):
    assert 0.0 < train_ratio < 1.0
    assert 0.0 <= val_ratio < 1.0
    assert train_ratio + val_ratio < 1.0

    groups = defaultdict(list)

    for item in data_dicts:
        if "vol_id" not in item:
            raise KeyError(
                "Each item must contain 'vol_id'. "
                "Add 'vol_id': vol_id in build_25d_pairs()."
            )

        groups[item["vol_id"]].append(item)

    volume_ids = list(groups.keys())

    rng = random.Random(seed)
    rng.shuffle(volume_ids)

    total_volumes = len(volume_ids)
    train_end = int(total_volumes * train_ratio)
    val_end = train_end + int(total_volumes * val_ratio)

    train_vols = set(volume_ids[:train_end])
    val_vols = set(volume_ids[train_end:val_end])
    test_vols = set(volume_ids[val_end:])

    train_dicts = []
    val_dicts = []
    test_dicts = []

    for vol_id, items in groups.items():
        items = sorted(items, key=lambda x: x["z_idx"])

        if vol_id in train_vols:
            train_dicts.extend(items)
        elif vol_id in val_vols:
            val_dicts.extend(items)
        else:
            test_dicts.extend(items)

    print(
        f"Volume Split -> "
        f"Train vols: {len(train_vols)} | "
        f"Val vols: {len(val_vols)} | "
        f"Test vols: {len(test_vols)}"
    )

    print(
        f"Slice Pairs -> "
        f"Train: {len(train_dicts)} | "
        f"Val: {len(val_dicts)} | "
        f"Test/Holdout: {len(test_dicts)}"
    )

    return train_dicts, val_dicts, test_dicts

In [ ]:
def natural_key(x):
    x = str(x)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", x)]

def list_tiff_files(folder):
    folder = Path(folder)
    files = [
        p for p in folder.iterdir()
        if p.suffix.lower() in [".tif", ".tiff"]
    ]
    return sorted(files, key=natural_key)

In [ ]:
def build_25d_pairs(noisy_dir, clean_dir, subset_name, window_size=5):
    noisy_files = list_tiff_files(noisy_dir)
    clean_files = list_tiff_files(clean_dir)
    print(f"{subset_name}: found {len(noisy_files)} noisy files")
    print(f"{subset_name}: found {len(clean_files)} clean files")

    clean_map = {p.name: p for p in clean_files}
    volume_groups = {}
    for p in noisy_files:
        match = re.search(r"(.+)_z(\d+)", p.name)
        if not match:
            raise ValueError(f"Filename {p.name} does not match expected '_z000' format.")

        vol_id = match.group(1)
        z_idx = int(match.group(2))
        volume_groups.setdefault(vol_id, []).append((z_idx, p))

    pairs = []
    half_w = window_size // 2

    for vol_id, slices in volume_groups.items():
        slices.sort(key=lambda x: x[0])
        num_slices = len(slices)

        for i in range(num_slices):
            z_idx, target_path = slices[i]
            target_name = target_path.name

            if target_name not in clean_map:
                raise ValueError(f"Missing clean match for {target_name}")

            gt_path = clean_map[target_name]
            neighbor_paths = []
            for offset in range(-half_w, half_w + 1):
                neighbor_i = i + offset
                neighbor_i = max(0, min(num_slices - 1, neighbor_i))
                neighbor_paths.append(str(slices[neighbor_i][1]))

            pairs.append({
                "noisy": neighbor_paths,
                "gt": str(gt_path),
                "id": f"{subset_name}_{vol_id}_z{z_idx:03d}",
                "filename": target_name,
                "vol_id": vol_id,
                "z_idx": z_idx,
                "subset": subset_name,
            })

    return pairs

### Read the dataset

In [ ]:
match dataset:
  case 'Nuclei':
    small_pairs = build_pairs(noisy_dir=NOISY_DATASET_SMALL, clean_dir=CLEAN_DATASET_SMALL, subset_name="small")
    large_pairs = build_pairs(noisy_dir=NOISY_DATASET_LARGE, clean_dir=CLEAN_DATASET_LARGE, subset_name="large")
    data_dicts = small_pairs + large_pairs
  case 'FMD':
    data_dicts = build_pairs(noisy_dir=NOISY_DATASET, clean_dir=CLEAN_DATASET, subset_name="fmd")
  case 'Planaria':
    data_dicts_small = build_25d_pairs(noisy_dir=NOISY_SMALL, clean_dir=CLEAN_SMALL, subset_name="planaria_small", window_size=5)
    data_dicts_large = build_25d_pairs(noisy_dir=NOISY_LARGE, clean_dir=CLEAN_LARGE, subset_name="planaria_large", window_size=5)
    data_dicts = data_dicts_small + data_dicts_large
  case 'Tribolium':
    data_dicts = build_25d_pairs(noisy_dir=NOISY_DATASET, clean_dir=CLEAN_DATASET, subset_name="tribolium", window_size=5)

# Evaluate

## For 2D

In [ ]:
def pad_to_multiple(x, multiple=16):
  b, c, h, w = x.shape

  pad_h = (multiple - h % multiple) % multiple
  pad_w = (multiple - w % multiple) % multiple
  if pad_h == 0 and pad_w == 0:
    return x, (h, w)
  x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
  return x, (h, w)


def crop_to_original(x, original_hw):
  h, w = original_hw
  return x[..., :h, :w]

In [ ]:
def model_forward(model, x, output_mode="clean"):
  y = model(x)

  if isinstance(y, (tuple, list)):
    y = y[0]
  if output_mode == "residual":
    y = x + y

  return y

In [ ]:
def predict_tta(model, x, output_mode="clean", tta_mode="x8"):
  if tta_mode is None or tta_mode == "none":
    return model_forward(model, x, output_mode=output_mode)

  dims = [2, 3]
  preds = []
  for k in range(4):
    x_aug = torch.rot90(x, k=k, dims=dims)
    y_aug = model_forward(model, x_aug, output_mode=output_mode)
    y = torch.rot90(y_aug, k=-k, dims=dims)
    preds.append(y)

  if tta_mode == "x8":
    for k in range(4):
      x_aug = torch.rot90(x, k=k, dims=dims)
      x_aug = torch.flip(x_aug, dims=[3])
      y_aug = model_forward(model, x_aug, output_mode=output_mode)
      y = torch.flip(y_aug, dims=[3])
      y = torch.rot90(y, k=-k, dims=dims)
      preds.append(y)

  return torch.stack(preds, dim=0).mean(dim=0)

In [ ]:
def evaluate_model(model, dataloader, device,output_mode="clean", tta_mode="x8", data_range=1.0, clamp=True, pad_multiple=16, use_sipsnr=True):
  model.eval()

  total_base_psnr = 0.0
  total_base_ssim = 0.0
  total_model_psnr = 0.0
  total_model_ssim = 0.0
  num_samples = 0

  if use_sipsnr and SampleSIPSNR is not None:
    base_sipsnr_metric = SampleSIPSNR(n_channels=1, use_scale_invariance=True).to(device)
    model_sipsnr_metric = SampleSIPSNR(n_channels=1, use_scale_invariance=True,).to(device)
  else:
    base_sipsnr_metric = None
    model_sipsnr_metric = None

  use_amp = device.type == "cuda"
  with torch.inference_mode():
    loop = tqdm(dataloader, desc="Evaluating validation set")

    for batch in loop:
      noisy_imgs = batch["noisy"].to(device, non_blocking=True).float()
      gt_imgs = batch["gt"].to(device, non_blocking=True).float()
      original_hw = noisy_imgs.shape[-2:]
      if pad_multiple is not None:
        noisy_input, original_hw = pad_to_multiple(noisy_imgs, multiple=pad_multiple)
      else:
        noisy_input = noisy_imgs

      with torch.amp.autocast(device_type=device.type, enabled=use_amp):
        preds = predict_tta(model=model, x=noisy_input, output_mode=output_mode, tta_mode=tta_mode)
        preds = crop_to_original(preds, original_hw)
        preds = preds.float()

        if clamp:
          preds = torch.clamp(preds, 0.0, 1.0)
          noisy_eval = torch.clamp(noisy_imgs, 0.0, 1.0)
          gt_eval = torch.clamp(gt_imgs, 0.0, 1.0)
        else:
          noisy_eval = noisy_imgs
          gt_eval = gt_imgs

        if base_sipsnr_metric is not None:
          base_sipsnr_metric.update(noisy_eval, gt_eval)
          model_sipsnr_metric.update(preds, gt_eval)

        preds_np = preds.detach().cpu().numpy()
        noisy_np = noisy_eval.detach().cpu().numpy()
        gt_np = gt_eval.detach().cpu().numpy()

        preds_np = preds_np[:, 0]
        noisy_np = noisy_np[:, 0]
        gt_np = gt_np[:, 0]

        for pred, raw, gt in zip(preds_np, noisy_np, gt_np):
          total_base_psnr += peak_signal_noise_ratio(gt, raw,data_range=data_range)
          total_base_ssim += structural_similarity(gt, raw,data_range=data_range)
          total_model_psnr += peak_signal_noise_ratio(gt, pred,data_range=data_range,)
          total_model_ssim += structural_similarity(gt, pred,data_range=data_range,)
          num_samples += 1

  avg_base_psnr = total_base_psnr / num_samples
  avg_base_ssim = total_base_ssim / num_samples
  avg_model_psnr = total_model_psnr / num_samples
  avg_model_ssim = total_model_ssim / num_samples
  if base_sipsnr_metric is not None:
    avg_base_sipsnr = base_sipsnr_metric.compute().mean().item()
    avg_model_sipsnr = model_sipsnr_metric.compute().mean().item()
  else:
    avg_base_sipsnr = None
    avg_model_sipsnr = None

  print("\n" + "=" * 60)
  print(f"FINAL EVALUATION ({num_samples} images/patches)")
  print("=" * 60)
  print("BASELINE: noisy input")
  print(f"  PSNR: {avg_base_psnr:.4f} dB")
  print(f"  SSIM: {avg_base_ssim:.5f}")

  if avg_base_sipsnr is not None:
    print(f"  SI-PSNR: {avg_base_sipsnr:.4f} dB")

  print("-" * 60)
  print(f"MODEL: output_mode={output_mode}, TTA={tta_mode}")
  print(f"  PSNR: {avg_model_psnr:.4f} dB")
  print(f"  SSIM: {avg_model_ssim:.5f}")

  if avg_model_sipsnr is not None:
    print(f"  SI-PSNR: {avg_model_sipsnr:.4f} dB")

  print("-" * 60)
  print("IMPROVEMENT")
  print(f"  PSNR: +{avg_model_psnr - avg_base_psnr:.4f} dB")
  print(f"  SSIM: +{avg_model_ssim - avg_base_ssim:.5f}")

  if avg_model_sipsnr is not None:
    print(f"  SI-PSNR: +{avg_model_sipsnr - avg_base_sipsnr:.4f} dB")

  print("=" * 60 + "\n")
  results = {
      "baseline_psnr": avg_base_psnr,
      "baseline_ssim": avg_base_ssim,
      "model_psnr": avg_model_psnr,
      "model_ssim": avg_model_ssim,
      "psnr_improvement": avg_model_psnr - avg_base_psnr,
      "ssim_improvement": avg_model_ssim - avg_base_ssim,
  }

  if avg_model_sipsnr is not None:
    results.update(
        {
            "baseline_sipsnr": avg_base_sipsnr,
            "model_sipsnr": avg_model_sipsnr,
            "sipsnr_improvement": avg_model_sipsnr - avg_base_sipsnr,
        }
    )

  return results

## For 3D

In [ ]:
def pad_to_multiple(x, multiple=16):
  b, c, h, w = x.shape

  pad_h = (multiple - h % multiple) % multiple
  pad_w = (multiple - w % multiple) % multiple
  if pad_h == 0 and pad_w == 0:
    return x, (h, w)
  x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
  return x, (h, w)


def crop_to_original(x, original_hw):
  h, w = original_hw
  return x[..., :h, :w]

In [ ]:
def model_forward(model, x, output_mode="clean"):
  y = model(x)

  if isinstance(y, (tuple, list)):
    y = y[0]
  if output_mode == "residual":
    y = x + y

  return y

In [ ]:
def predict_tta(model, x, output_mode="clean", tta_mode="x8"):
  if tta_mode is None or tta_mode == "none":
    return model_forward(model, x, output_mode=output_mode)

  dims = [2, 3]
  preds = []
  for k in range(4):
    x_aug = torch.rot90(x, k=k, dims=dims)
    y_aug = model_forward(model, x_aug, output_mode=output_mode)
    y = torch.rot90(y_aug, k=-k, dims=dims)
    preds.append(y)

  if tta_mode == "x8":
    for k in range(4):
      x_aug = torch.rot90(x, k=k, dims=dims)
      x_aug = torch.flip(x_aug, dims=[3])
      y_aug = model_forward(model, x_aug, output_mode=output_mode)
      y = torch.flip(y_aug, dims=[3])
      y = torch.rot90(y, k=-k, dims=dims)
      preds.append(y)

  return torch.stack(preds, dim=0).mean(dim=0)

In [ ]:
def evaluate_model_25d(
    model,
    dataloader,
    device,
    output_mode="clean",
    tta_mode="x4",
    data_range=1.0,
    clamp=True,
    pad_multiple=16,
    use_sipsnr=True,
    use_amp=False,
):
    model.eval()

    total_base_psnr = 0.0
    total_base_ssim = 0.0
    total_model_psnr = 0.0
    total_model_ssim = 0.0
    num_samples = 0

    if use_sipsnr and SampleSIPSNR is not None:
        base_sipsnr_metric = SampleSIPSNR(
            n_channels=1,
            use_scale_invariance=True,
        ).to(device)

        model_sipsnr_metric = SampleSIPSNR(
            n_channels=1,
            use_scale_invariance=True,
        ).to(device)
    else:
        base_sipsnr_metric = None
        model_sipsnr_metric = None

    with torch.inference_mode():
        loop = tqdm(dataloader, desc="Evaluating 2.5D validation set")

        for batch in loop:
            noisy_imgs = batch["noisy"].to(device, non_blocking=True).float()
            gt_imgs = batch["gt"].to(device, non_blocking=True).float()

            original_hw = noisy_imgs.shape[-2:]

            if pad_multiple is not None:
                noisy_input, original_hw = pad_to_multiple(
                    noisy_imgs,
                    multiple=pad_multiple,
                )
            else:
                noisy_input = noisy_imgs

            with torch.amp.autocast(
                device_type=device.type,
                enabled=use_amp,
            ):
                preds = predict_tta(
                    model=model,
                    x=noisy_input,
                    output_mode=output_mode,
                    tta_mode=tta_mode,
                )

            preds = crop_to_original(preds, original_hw)
            preds = preds.float()

            if not torch.isfinite(preds).all():
                print("Warning: NaN/Inf found in validation predictions. Replacing with finite values.")
                preds = torch.nan_to_num(
                    preds,
                    nan=0.0,
                    posinf=1.0,
                    neginf=0.0,
                )

            if clamp:
                preds = torch.clamp(preds, 0.0, 1.0)
                noisy_eval = torch.clamp(noisy_imgs, 0.0, 1.0)
                gt_eval = torch.clamp(gt_imgs, 0.0, 1.0)
            else:
                noisy_eval = noisy_imgs
                gt_eval = gt_imgs

            center_idx = noisy_eval.shape[1] // 2
            noisy_eval_center = noisy_eval[:, center_idx:center_idx + 1, :, :]

            if base_sipsnr_metric is not None:
                base_sipsnr_metric.update(noisy_eval_center, gt_eval)
                model_sipsnr_metric.update(preds, gt_eval)

            preds_np = preds.detach().cpu().numpy()[:, 0]
            noisy_np_center = noisy_eval_center.detach().cpu().numpy()[:, 0]
            gt_np = gt_eval.detach().cpu().numpy()[:, 0]

            for pred, raw, gt in zip(preds_np, noisy_np_center, gt_np):
                if not np.isfinite(pred).all():
                    pred = np.nan_to_num(pred, nan=0.0, posinf=1.0, neginf=0.0)

                total_base_psnr += peak_signal_noise_ratio(
                    gt,
                    raw,
                    data_range=data_range,
                )

                total_base_ssim += structural_similarity(
                    gt,
                    raw,
                    data_range=data_range,
                )

                total_model_psnr += peak_signal_noise_ratio(
                    gt,
                    pred,
                    data_range=data_range,
                )

                total_model_ssim += structural_similarity(
                    gt,
                    pred,
                    data_range=data_range,
                )

                num_samples += 1

    avg_base_psnr = total_base_psnr / max(num_samples, 1)
    avg_base_ssim = total_base_ssim / max(num_samples, 1)

    avg_model_psnr = total_model_psnr / max(num_samples, 1)
    avg_model_ssim = total_model_ssim / max(num_samples, 1)

    if base_sipsnr_metric is not None:
        avg_base_sipsnr = base_sipsnr_metric.compute().mean().item()
        avg_model_sipsnr = model_sipsnr_metric.compute().mean().item()
    else:
        avg_base_sipsnr = None
        avg_model_sipsnr = None

    print("\n" + "=" * 60)
    print(f"FINAL 2.5D EVALUATION ({num_samples} images/patches)")
    print("=" * 60)

    print("BASELINE: center noisy slice")
    print(f"  PSNR: {avg_base_psnr:.4f} dB")
    print(f"  SSIM: {avg_base_ssim:.5f}")

    if avg_base_sipsnr is not None:
        print(f"  SI-PSNR: {avg_base_sipsnr:.4f} dB")

    print("-" * 60)

    print(f"MODEL: output_mode={output_mode}, TTA={tta_mode}")
    print(f"  PSNR: {avg_model_psnr:.4f} dB")
    print(f"  SSIM: {avg_model_ssim:.5f}")

    if avg_model_sipsnr is not None:
        print(f"  SI-PSNR: {avg_model_sipsnr:.4f} dB")

    print("-" * 60)

    print("IMPROVEMENT")
    print(f"  PSNR: +{avg_model_psnr - avg_base_psnr:.4f} dB")
    print(f"  SSIM: +{avg_model_ssim - avg_base_ssim:.5f}")

    if avg_model_sipsnr is not None:
        print(f"  SI-PSNR: +{avg_model_sipsnr - avg_base_sipsnr:.4f} dB")

    print("=" * 60 + "\n")

    results = {
        "baseline_psnr": avg_base_psnr,
        "baseline_ssim": avg_base_ssim,
        "model_psnr": avg_model_psnr,
        "model_ssim": avg_model_ssim,
        "psnr_improvement": avg_model_psnr - avg_base_psnr,
        "ssim_improvement": avg_model_ssim - avg_base_ssim,
    }

    if avg_model_sipsnr is not None:
        results.update(
            {
                "baseline_sipsnr": avg_base_sipsnr,
                "model_sipsnr": avg_model_sipsnr,
                "sipsnr_improvement": avg_model_sipsnr - avg_base_sipsnr,
            }
        )

    return results

# Plot

## 2D

In [ ]:
def visualize_2d_nuclei(model, dataloader, device, num_samples=3, output_mode="clean", tta_mode="x4", clamp=True, pad_multiple=16, cmap="gray", show_error=True):
  model.eval()
  device_type = device.type if isinstance(device, torch.device) else str(device)
  use_amp = device_type == "cuda"
  plotted = 0

  with torch.inference_mode():
    for batch in dataloader:
      noisy_imgs = batch["noisy"].to(device, non_blocking=True).float()
      gt_imgs = batch["gt"].to(device, non_blocking=True).float()

      original_hw = noisy_imgs.shape[-2:]

      if pad_multiple is not None:
        model_input, original_hw = pad_to_multiple(noisy_imgs, multiple=pad_multiple)
      else:
        model_input = noisy_imgs

      with torch.amp.autocast(device_type=device_type, enabled=use_amp):
        preds = predict_tta(model=model, x=model_input, output_mode=output_mode, tta_mode=tta_mode)

      preds = crop_to_original(preds, original_hw)
      preds = preds.float()

      if clamp:
        preds = torch.clamp(preds, 0.0, 1.0)
        noisy_imgs = torch.clamp(noisy_imgs, 0.0, 1.0)
        gt_imgs = torch.clamp(gt_imgs, 0.0, 1.0)

      noisy_np = noisy_imgs.detach().cpu().numpy()
      preds_np = preds.detach().cpu().numpy()
      gt_np = gt_imgs.detach().cpu().numpy()
      batch_size = noisy_np.shape[0]
      for i in range(batch_size):
        if plotted >= num_samples:
          return

        n_plot = noisy_np[i, 0]
        p_plot = preds_np[i, 0]
        g_plot = gt_np[i, 0]

        base_psnr = peak_signal_noise_ratio(g_plot, n_plot, data_range=1.0)
        pred_psnr = peak_signal_noise_ratio(g_plot, p_plot, data_range=1.0)
        base_ssim = structural_similarity(g_plot, n_plot, data_range=1.0)
        pred_ssim = structural_similarity(g_plot, p_plot, data_range=1.0)

        if show_error:
          fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        else:
          fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        vmin, vmax = 0.0, 1.0
        axes[0].imshow(n_plot, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[0].set_title(
            f"Noisy Input\nPSNR: {base_psnr:.2f} | SSIM: {base_ssim:.4f}",
            fontsize=12,
        )
        axes[0].axis("off")
        axes[1].imshow(p_plot, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[1].set_title(
            f"Model Output\nPSNR: {pred_psnr:.2f} | SSIM: {pred_ssim:.4f}",
            fontsize=12,
        )
        axes[1].axis("off")
        axes[2].imshow(g_plot, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[2].set_title("Ground Truth", fontsize=12)
        axes[2].axis("off")
        if show_error:
          error = np.abs(p_plot - g_plot)
          im = axes[3].imshow(error, cmap="magma")
          axes[3].set_title(
              f"|Prediction - GT|\nMean error: {error.mean():.5f}",
              fontsize=12,
          )
          axes[3].axis("off")
        fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
        fig.suptitle(
            f"Sample {plotted + 1} | Output mode: {output_mode} | TTA: {tta_mode}",
            fontsize=15,
        )

        plt.tight_layout()
        plt.show()
        plotted += 1

## 3D

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm import tqdm


def visualize_25d_embryos(
    model,
    dataloader,
    device,
    num_samples=3,
    output_mode="clean",
    tta_mode="x4",
    clamp=True,
    pad_multiple=16,
    cmap="gray",
    show_error=True,
    use_amp=False,
    show_context=False,
):
    model.eval()

    device_type = device.type if isinstance(device, torch.device) else str(device)
    plotted = 0

    with torch.inference_mode():
        for batch in dataloader:
            noisy_imgs = batch["noisy"].to(device, non_blocking=True).float()
            gt_imgs = batch["gt"].to(device, non_blocking=True).float()

            original_hw = noisy_imgs.shape[-2:]

            if pad_multiple is not None:
                model_input, original_hw = pad_to_multiple(
                    noisy_imgs,
                    multiple=pad_multiple,
                )
            else:
                model_input = noisy_imgs

            with torch.amp.autocast(
                device_type=device_type,
                enabled=use_amp,
            ):
                preds = predict_tta(
                    model=model,
                    x=model_input,
                    output_mode=output_mode,
                    tta_mode=tta_mode,
                )

            preds = crop_to_original(preds, original_hw)
            preds = preds.float()

            if not torch.isfinite(preds).all():
                print("Warning: NaN/Inf found in predictions. Replacing with finite values.")
                preds = torch.nan_to_num(
                    preds,
                    nan=0.0,
                    posinf=1.0,
                    neginf=0.0,
                )

            if clamp:
                preds = torch.clamp(preds, 0.0, 1.0)
                noisy_imgs = torch.clamp(noisy_imgs, 0.0, 1.0)
                gt_imgs = torch.clamp(gt_imgs, 0.0, 1.0)

            noisy_np = noisy_imgs.detach().cpu().numpy()
            preds_np = preds.detach().cpu().numpy()
            gt_np = gt_imgs.detach().cpu().numpy()

            batch_size = noisy_np.shape[0]

            for i in range(batch_size):
                if plotted >= num_samples:
                    return

                center_idx = noisy_np.shape[1] // 2

                n_plot = noisy_np[i, center_idx]
                p_plot = preds_np[i, 0]
                g_plot = gt_np[i, 0]

                n_plot = np.nan_to_num(n_plot, nan=0.0, posinf=1.0, neginf=0.0)
                p_plot = np.nan_to_num(p_plot, nan=0.0, posinf=1.0, neginf=0.0)
                g_plot = np.nan_to_num(g_plot, nan=0.0, posinf=1.0, neginf=0.0)

                base_psnr = peak_signal_noise_ratio(
                    g_plot,
                    n_plot,
                    data_range=1.0,
                )
                pred_psnr = peak_signal_noise_ratio(
                    g_plot,
                    p_plot,
                    data_range=1.0,
                )

                base_ssim = structural_similarity(
                    g_plot,
                    n_plot,
                    data_range=1.0,
                )
                pred_ssim = structural_similarity(
                    g_plot,
                    p_plot,
                    data_range=1.0,
                )

                if show_context:
                    num_cols = noisy_np.shape[1] + 3 if show_error else noisy_np.shape[1] + 2
                    fig, axes = plt.subplots(1, num_cols, figsize=(4 * num_cols, 5))

                    for c in range(noisy_np.shape[1]):
                        axes[c].imshow(noisy_np[i, c], cmap=cmap, vmin=0.0, vmax=1.0)
                        if c == center_idx:
                            axes[c].set_title(f"Input z center\nchannel {c}", fontsize=11)
                        else:
                            axes[c].set_title(f"Input channel {c}", fontsize=11)
                        axes[c].axis("off")

                    pred_ax = noisy_np.shape[1]
                    gt_ax = noisy_np.shape[1] + 1

                    axes[pred_ax].imshow(p_plot, cmap=cmap, vmin=0.0, vmax=1.0)
                    axes[pred_ax].set_title(
                        f"Model Output\nPSNR: {pred_psnr:.2f} | SSIM: {pred_ssim:.4f}",
                        fontsize=11,
                    )
                    axes[pred_ax].axis("off")

                    axes[gt_ax].imshow(g_plot, cmap=cmap, vmin=0.0, vmax=1.0)
                    axes[gt_ax].set_title("Ground Truth", fontsize=11)
                    axes[gt_ax].axis("off")

                    if show_error:
                        err_ax = noisy_np.shape[1] + 2
                        error = np.abs(p_plot - g_plot)
                        im = axes[err_ax].imshow(error, cmap="magma")
                        axes[err_ax].set_title(
                            f"|Prediction - GT|\nMean: {error.mean():.5f}",
                            fontsize=11,
                        )
                        axes[err_ax].axis("off")
                        fig.colorbar(im, ax=axes[err_ax], fraction=0.046, pad=0.04)

                else:
                    if show_error:
                        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
                    else:
                        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

                    axes[0].imshow(n_plot, cmap=cmap, vmin=0.0, vmax=1.0)
                    axes[0].set_title(
                        f"Noisy Center Slice\nPSNR: {base_psnr:.2f} | SSIM: {base_ssim:.4f}",
                        fontsize=12,
                    )
                    axes[0].axis("off")

                    axes[1].imshow(p_plot, cmap=cmap, vmin=0.0, vmax=1.0)
                    axes[1].set_title(
                        f"Model Output\nPSNR: {pred_psnr:.2f} | SSIM: {pred_ssim:.4f}",
                        fontsize=12,
                    )
                    axes[1].axis("off")

                    axes[2].imshow(g_plot, cmap=cmap, vmin=0.0, vmax=1.0)
                    axes[2].set_title("Ground Truth", fontsize=12)
                    axes[2].axis("off")

                    if show_error:
                        error = np.abs(p_plot - g_plot)
                        im = axes[3].imshow(error, cmap="magma")
                        axes[3].set_title(
                            f"|Prediction - GT|\nMean error: {error.mean():.5f}",
                            fontsize=12,
                        )
                        axes[3].axis("off")
                        fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

                fig.suptitle(
                    f"2.5D Sample {plotted + 1} | "
                    f"Output mode: {output_mode} | TTA: {tta_mode}",
                    fontsize=15,
                )

                plt.tight_layout()
                plt.show()

                plotted += 1

# Custom Model

In [ ]:
class SimpleGate(nn.Module):
  def forward(self, x):
    x1, x2 = x.chunk(2, dim=1)
    return x1 * x2

In [ ]:
class LayerNorm2d(nn.Module):
  def __init__(self, channels, eps=1e-3):
    super().__init__()
    self.norm = nn.LayerNorm(channels, eps=eps)

  def forward(self, x):
    x = x.permute(0, 2, 3, 1).contiguous()
    x = self.norm(x)
    x = x.permute(0, 3, 1, 2).contiguous()
    return x

In [ ]:
class SimplifiedChannelAttention(nn.Module):
  def __init__(self, channels):
    super().__init__()
    self.pool = nn.AdaptiveAvgPool2d(1)
    self.conv = nn.Conv2d(channels, channels, kernel_size=1, padding=0)

  def forward(self, x):
    return x * self.conv(self.pool(x))

In [ ]:
class NAFBlock(nn.Module):
  def __init__(self, dim, dw_expand=2, ffn_expand=2):
    super().__init__()
    dw_channels = dim * dw_expand
    ffn_channels = dim * ffn_expand
    self.norm1 = LayerNorm2d(dim)
    self.conv1 = nn.Conv2d(dim, dw_channels, kernel_size=1, padding=0)
    self.conv2 = nn.Conv2d(
        dw_channels,
        dw_channels,
        kernel_size=3,
        padding=1,
        groups=dw_channels,
    )
    self.sg = SimpleGate()
    self.sca = SimplifiedChannelAttention(dw_channels // 2)
    self.conv3 = nn.Conv2d(dw_channels // 2, dim, kernel_size=1, padding=0)
    self.norm2 = LayerNorm2d(dim)
    self.conv4 = nn.Conv2d(dim, ffn_channels, kernel_size=1, padding=0)
    self.conv5 = nn.Conv2d(ffn_channels // 2, dim, kernel_size=1, padding=0)
    self.beta = nn.Parameter(torch.zeros(1, dim, 1, 1))
    self.gamma = nn.Parameter(torch.zeros(1, dim, 1, 1))

  def forward(self, x):
    identity = x
    y = self.norm1(x)
    y = self.conv1(y)
    y = self.conv2(y)
    y = self.sg(y)
    y = self.sca(y)
    y = self.conv3(y)
    x = identity + y * self.beta
    identity = x
    y = self.norm2(x)
    y = self.conv4(y)
    y = self.sg(y)
    y = self.conv5(y)
    x = identity + y * self.gamma

    return x

In [ ]:
class NAFStack(nn.Module):
  def __init__(self, dim, num_blocks):
    super().__init__()
    self.blocks = nn.Sequential(
        *[NAFBlock(dim) for _ in range(num_blocks)]
    )

  def forward(self, x):
    return self.blocks(x)

In [ ]:
class UpBlock(nn.Module):
  def __init__(self, in_channels, skip_channels, out_channels, num_blocks=1):
    super().__init__()
    self.up_conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.fuse = nn.Conv2d(
        out_channels + skip_channels,
        out_channels,
        kernel_size=1,
        padding=0,
    )
    self.refine = NAFStack(out_channels, num_blocks)

  def forward(self, x, skip):
    x = F.interpolate(
        x,
        size=skip.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )
    x = self.up_conv(x)
    x = torch.cat([x, skip], dim=1)
    x = self.fuse(x)
    x = self.refine(x)

    return x

In [ ]:
class CustomSOTAHybrid(nn.Module):
  def __init__(self,in_channels=1, out_channels=1, base_dim=32, crop_size=256, use_mamba=True):
    super().__init__()
    self.in_channels = in_channels
    self.out_channels = out_channels
    self.base_dim = base_dim
    self.crop_size = crop_size
    self.use_mamba = use_mamba

    c1 = base_dim
    c2 = base_dim * 2
    c3 = base_dim * 4
    c4 = base_dim * 8

    self.embed = nn.Conv2d(
        in_channels,
        c1,
        kernel_size=3,
        padding=1,
    )

    self.enc1 = NAFStack(c1, num_blocks=2)
    self.down1 = nn.Conv2d(c1, c2, kernel_size=2, stride=2)
    self.enc2 = NAFStack(c2, num_blocks=2)
    self.down2 = nn.Conv2d(c2, c3, kernel_size=2, stride=2)
    self.enc3 = NAFStack(c3, num_blocks=3)
    self.down3 = nn.Conv2d(c3, c4, kernel_size=2, stride=2)

    bottleneck_img_size = crop_size // 8

    if use_mamba:
      self.bottleneck = MambaIRv2(
          img_size=bottleneck_img_size,
          patch_size=1,
          in_chans=c4,
          embed_dim=c4,
          depths=(4, 4),
          mlp_ratio=2.0,
          upscale=1,
          img_range=1.0,
          upsampler="",
          resi_connection="1conv",
      )
    else:
      self.bottleneck = NAFStack(c4, num_blocks=6)

    self.bottleneck_refine = NAFStack(c4, num_blocks=2)

    self.up1 = UpBlock(
        in_channels=c4,
        skip_channels=c3,
        out_channels=c3,
        num_blocks=2,
    )

    self.up2 = UpBlock(
        in_channels=c3,
        skip_channels=c2,
        out_channels=c2,
        num_blocks=2,
    )

    self.up3 = UpBlock(
        in_channels=c2,
        skip_channels=c1,
        out_channels=c1,
        num_blocks=2,
    )

    self.output_projection = nn.Conv2d(
        c1,
        out_channels,
        kernel_size=3,
        padding=1,
    )

  def forward(self, x):
    noisy_reference = x[:, 0:1, :, :]
    f0 = self.embed(x)
    skip1 = self.enc1(f0)
    f1 = self.down1(skip1)
    skip2 = self.enc2(f1)
    f2 = self.down2(skip2)
    skip3 = self.enc3(f2)
    f3 = self.down3(skip3)
    bot = self.bottleneck(f3)
    if bot.shape[-2:] != f3.shape[-2:]:
      bot = F.interpolate(
          bot,
          size=f3.shape[-2:],
          mode="bilinear",
          align_corners=False,
      )
    bot = self.bottleneck_refine(bot)
    d1 = self.up1(bot, skip3)
    d2 = self.up2(d1, skip2)
    d3 = self.up3(d2, skip1)
    predicted_residual = self.output_projection(d3)
    denoised = noisy_reference + predicted_residual

    return denoised

# Custom Loss Function

In [ ]:
class CharbonnierMSEEdgeLoss(nn.Module):
  def __init__(self, eps=1e-3, charbonnier_weight=0.90, mse_weight=0.05, edge_weight=0.05,):

    super().__init__()
    self.eps = eps
    self.charbonnier_weight = charbonnier_weight
    self.mse_weight = mse_weight
    self.edge_weight = edge_weight

  def forward(self, pred, target):
    diff = pred - target
    charbonnier_loss = torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))
    mse_loss = F.mse_loss(pred, target)

    pred_dx = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    pred_dy = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    target_dx = target[:, :, :, 1:] - target[:, :, :, :-1]
    target_dy = target[:, :, 1:, :] - target[:, :, :-1, :]

    edge_loss_x = F.l1_loss(pred_dx, target_dx)
    edge_loss_y = F.l1_loss(pred_dy, target_dy)
    edge_loss = 0.5 * (edge_loss_x + edge_loss_y)

    loss = (
        self.charbonnier_weight * charbonnier_loss
        + self.mse_weight * mse_loss
        + self.edge_weight * edge_loss
        )

    return loss

# 3K Fold

## 2D

In [ ]:
def get_2d_kfold_splits(data_dicts, n_splits=3, seed=42):
  data_dicts = list(data_dicts)
  kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

  fold_splits = []
  for fold, (train_idx, val_idx) in enumerate(kf.split(data_dicts)):
    train_dicts = [data_dicts[i] for i in train_idx]
    val_dicts = [data_dicts[i] for i in val_idx]
    fold_splits.append((train_dicts, val_dicts))
    print(
        f"Fold {fold + 1} Split -> "
        f"Train Images: {len(train_dicts)} | "
        f"Val Images: {len(val_dicts)}"
        )

  return fold_splits

## 3D

In [ ]:
def get_group_kfold_splits(data_dicts, group_key="vol_id", n_splits=3):
  data_dicts = list(data_dicts)
  groups = [item[group_key] for item in data_dicts]
  gkf = GroupKFold(n_splits=n_splits)

  fold_splits = []
  for fold, (train_idx, val_idx) in enumerate(gkf.split(data_dicts, groups=groups)):
    train_dicts = [data_dicts[i] for i in train_idx]
    val_dicts = [data_dicts[i] for i in val_idx]

    train_groups = set(item[group_key] for item in train_dicts)
    val_groups = set(item[group_key] for item in val_dicts)

    assert train_groups.isdisjoint(val_groups), "Group leakage detected!"

    fold_splits.append((train_dicts, val_dicts))

    print(
        f"Fold {fold + 1} Split -> "
        f"Train Samples: {len(train_dicts)} | "
        f"Val Samples: {len(val_dicts)} | "
        f"Train Groups: {len(train_groups)} | "
        f"Val Groups: {len(val_groups)}"
    )

  return fold_splits

# Main Loop

## 2D

In [ ]:
train_transforms = Compose(
    [
        CustomTiffLoader(keys=("noisy", "gt")),
        IndependentPercentileScaler(keys=("noisy", "gt"), lower=0.1, upper=99.9, clip=True),
        AlbumentationsDictTransform(keys=("noisy", "gt"), crop_size=256, min_patch_std=0.005, max_tries=10),
    ]
)

eval_transforms = Compose(
    [
        CustomTiffLoader(keys=("noisy", "gt")),
        IndependentPercentileScaler(keys=("noisy", "gt"), lower=0.1, upper=99.9, clip=True),
        ToTensor2D(keys=("noisy", "gt")),
    ]
)

print("Custom Transforms defined successfully.")

In [ ]:
n_folds = 3
fold_splits = get_2d_kfold_splits(data_dicts, n_splits=n_folds, seed=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

num_epochs = 100
steps_per_epoch = 400
validate_every = 5
use_amp = False
max_val_samples = 250

In [ ]:
fold_best_psnrs = []
fold_best_sipsnrs = []

for fold in range(n_folds):
  print("\n" + "=" * 60)
  print(f" INITIATING FOLD {fold + 1} / {n_folds}")
  print("=" * 60)

  train_dicts, val_dicts = fold_splits[fold]
  print(f"Train images: {len(train_dicts)} | Val images: {len(val_dicts)}")

  train_ds = Dataset(data=train_dicts, transform=train_transforms)
  val_ds = CacheDataset(data=val_dicts, transform=eval_transforms, cache_rate=1.0, num_workers=0)

  torch.manual_seed(42)
  num_val_samples = min(max_val_samples, len(val_ds))
  subset_indices = torch.randperm(len(val_ds))[:num_val_samples].tolist()
  val_subset = Subset(val_ds, subset_indices)

  train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=False, drop_last=True)
  val_loader_fast = DataLoader(val_subset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)
  val_loader_full = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

  print(f"Optimized Validation Set (Epochs): {len(val_loader_fast.dataset)} images")

  model = CustomSOTAHybrid(
      in_channels=1,
      out_channels=1,
      base_dim=32,
      crop_size=256,
      use_mamba=True,
  ).to(device)

  criterion = CharbonnierMSEEdgeLoss(charbonnier_weight=0.70, mse_weight=0.25, edge_weight=0.01).to(device)

  optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
  scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)

  best_score = -1e9
  best_val_psnr = -1e9
  best_val_sipsnr = None

  save_path = f"custom_sota_1channel_fold{fold+1}_best.pth"
  latest_path = f"custom_sota_1channel_fold{fold+1}_latest.pth"
  os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)

  for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    actual_steps = 0
    train_iter = iter(train_loader)

    train_loop = tqdm(range(steps_per_epoch), desc=f"Fold {fold+1} - Epoch {epoch + 1}/{num_epochs} [TRAIN]", mininterval=5.0)

    for step in train_loop:
      try:
        batch = next(train_iter)
      except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

      noisy = batch["noisy"].to(device, non_blocking=True).float()
      gt = batch["gt"].to(device, non_blocking=True).float()

      optimizer.zero_grad(set_to_none=True)

      with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_amp):
        padded_noisy, hw = pad_to_multiple(noisy, 16)
        padded_noisy = padded_noisy + 1e-6

        predictions = model(padded_noisy)
        predictions = crop_to_original(predictions, hw)

        loss = criterion(predictions, gt)

      scaler.scale(loss).backward()
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      scaler.step(optimizer)
      scaler.update()

      train_loss += loss.item()
      actual_steps += 1
      train_loop.set_postfix(loss=train_loss / actual_steps, lr=optimizer.param_groups[0]["lr"])

    avg_train_loss = train_loss / max(actual_steps, 1)
    scheduler.step()

    torch.save({
        "epoch": epoch + 1,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
    }, latest_path)

    if (epoch + 1) % validate_every == 0 or epoch == num_epochs - 1:
        metrics = evaluate_model(
            model=model, dataloader=val_loader_fast, device=device,
            output_mode="clean", tta_mode="x4", data_range=1.0, clamp=True, pad_multiple=16, use_sipsnr=True,
        )

        val_psnr = metrics["model_psnr"]
        val_sipsnr = metrics.get("model_sipsnr", None)

        score = val_sipsnr if val_sipsnr is not None else val_psnr
        score_name = "SI-PSNR" if val_sipsnr is not None else "PSNR"

        if score > best_score:
          best_score = score
          best_val_psnr = val_psnr
          best_val_sipsnr = val_sipsnr

          checkpoint = {
              "epoch": epoch + 1,
              "model": model.state_dict(),
              "best_val_psnr": best_val_psnr,
              "best_val_sipsnr": best_val_sipsnr,
          }
          torch.save(checkpoint, save_path)
          print(f"--> Best Fold {fold+1} model saved! ({score_name}: {best_score:.4f})")

  print(f"\n--- Fold {fold + 1} Training Complete. Executing Strict Re-Evaluation ---")
  best_checkpoint = torch.load(save_path, map_location=device)
  model.load_state_dict(best_checkpoint["model"])
  model.eval()

  final_fold_metrics = evaluate_model(
      model=model, dataloader=val_loader_full, device=device,
      output_mode="clean", tta_mode="x8", data_range=1.0, clamp=True, pad_multiple=16, use_sipsnr=True
  )
  final_val_psnr = final_fold_metrics["model_psnr"]
  final_val_sipsnr = final_fold_metrics.get("model_sipsnr", None)
  fold_best_psnrs.append(final_val_psnr)
  if final_val_sipsnr is not None:
    fold_best_sipsnrs.append(final_val_sipsnr)

  print(f" Fold {fold + 1} Official PSNR: {final_val_psnr:.4f} dB | SI-PSNR: {final_val_sipsnr:.4f} dB")
  if fold == 0:
    print(f"\nGenerating qualitative visual triplets for Fold 1...")
    visualize_2d_nuclei(
        model=model, dataloader=val_loader_full, device=device,
        num_samples=3, output_mode="clean", tta_mode="x8", clamp=True, pad_multiple=16, cmap="viridis", show_error=True
    )

print("\n" + "=" * 60)
print("FINAL 3-FOLD CROSS-VALIDATION SUMMARY")
print("=" * 60)
print(f"Mean PSNR:    {np.mean(fold_best_psnrs):.4f} ± {np.std(fold_best_psnrs):.4f} dB")
if fold_best_sipsnrs:
  print(f"Mean SI-PSNR: {np.mean(fold_best_sipsnrs):.4f} ± {np.std(fold_best_sipsnrs):.4f} dB")
print("=" * 60)

## 3D

In [ ]:
for d in data_dicts:
  if "vol_id" not in d:
    d["vol_id"] = "_".join(d["id"].split("_")[:-1])

In [ ]:
n_folds = 3
fold_splits = get_group_kfold_splits(data_dicts, group_key="vol_id", n_splits=n_folds)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

num_epochs = 100
steps_per_epoch = 400
validate_every = 5
use_amp = False

In [ ]:
train_transforms_25d = Compose([
    CustomTiffLoader25D(keys=("noisy", "gt")),

    PairedPercentileScaler25D(
        keys=("noisy", "gt"),
        reference_key="noisy",
        lower=0.1,
        upper=99.9,
        clip=True,
        save_stats=False,
    ),

    AlbumentationsDictTransform25D(
        keys=("noisy", "gt"),
        crop_size=256,
        min_patch_std=0.005,
        max_tries=10,
    ),
])

eval_transforms_25d = Compose([
    CustomTiffLoader25D(keys=("noisy", "gt")),

    PairedPercentileScaler25D(
        keys=("noisy", "gt"),
        reference_key="noisy",
        lower=0.1,
        upper=99.9,
        clip=True,
        save_stats=True,
    ),

    ToTensor25D(keys=("noisy", "gt")),
])

In [ ]:
fold_best_psnrs = []
fold_best_sipsnrs = []

for fold in range(n_folds):
  print("\n" + "=" * 80)
  print(f"INITIATING FOLD {fold + 1} / {n_folds} (2.5D Tribolium)")
  print("=" * 80)

  train_dicts, val_dicts = fold_splits[fold]
  print(f"Dataset Split -> Train: {len(train_dicts)} slices | Full Val: {len(val_dicts)} slices")

  train_ds = Dataset(data=train_dicts, transform=train_transforms_25d)
  val_ds = CacheDataset(data=val_dicts, transform=eval_transforms_25d, cache_rate=0.0, num_workers=0)

  torch.manual_seed(42)
  num_val_samples = min(250, len(val_ds))
  subset_indices = torch.randperm(len(val_ds))[:num_val_samples].tolist()
  val_subset = Subset(val_ds, subset_indices)

  num_val_full = min(2000, len(val_ds))
  full_indices = torch.randperm(len(val_ds))[:num_val_full].tolist()
  val_subset_full = Subset(val_ds, full_indices)

  train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0, pin_memory=False, drop_last=True)
  val_loader_fast = DataLoader(val_subset, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)

  val_loader_full = DataLoader(val_subset_full, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)
  print(f"Optimized Validation Set (Epochs): {len(val_loader_fast.dataset)} images")

  model = CustomSOTAHybrid(
      in_channels=5,
      out_channels=1,
      base_dim=32,
      crop_size=256,
      use_mamba=True,
  ).to(device)

  criterion = CharbonnierMSEEdgeLoss(charbonnier_weight=0.70, mse_weight=0.25, edge_weight=0.01).to(device)
  optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
  scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
  scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)

  best_score = -1e9
  best_val_psnr = -1e9
  best_val_sipsnr = None

  checkpoint_dir = "/content/drive/MyDrive/3 Semester/Thesis/Checkpoints/2.5D/"
  os.makedirs(checkpoint_dir, exist_ok=True)

  save_path = f"{checkpoint_dir}custom_sota_25d_fold{fold+1}_best.pth"
  latest_path = f"{checkpoint_dir}custom_sota_25d_fold{fold+1}_latest.pth"
  os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)

  for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    actual_steps = 0
    train_iter = iter(train_loader)
    train_loop = tqdm(range(steps_per_epoch), desc=f"Fold {fold+1} - Epoch {epoch + 1}/{num_epochs} [TRAIN]", mininterval=5.0)

    for step in train_loop:
      try:
        batch = next(train_iter)
      except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

      noisy = batch["noisy"].to(device, non_blocking=True).float()
      gt = batch["gt"].to(device, non_blocking=True).float()

      optimizer.zero_grad(set_to_none=True)

      with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_amp):
        padded_noisy, hw = pad_to_multiple(noisy, 16)

        predictions = model(padded_noisy)
        predictions = crop_to_original(predictions, hw)
        loss = criterion(predictions, gt)

      scaler.scale(loss).backward()
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      scaler.step(optimizer)
      scaler.update()

      train_loss += loss.item()
      actual_steps += 1
      train_loop.set_postfix(loss=train_loss / actual_steps, lr=optimizer.param_groups[0]["lr"])

    avg_train_loss = train_loss / max(actual_steps, 1)
    scheduler.step()


    torch.save({
        "epoch": epoch + 1,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_score": best_score,
    }, latest_path)


    if (epoch + 1) % validate_every == 0:
      print("\nTriggering fast subset validation...")
      metrics = evaluate_model_25d(
          model=model, dataloader=val_loader_fast, device=device,
          output_mode="clean", tta_mode="x4", data_range=1.0, clamp=True, pad_multiple=16, use_sipsnr=True, use_amp=False
      )

      val_psnr = metrics["model_psnr"]
      val_sipsnr = metrics.get("model_sipsnr", None)
      score = val_sipsnr if val_sipsnr is not None else val_psnr
      score_name = "SI-PSNR" if val_sipsnr is not None else "PSNR"

      if score > best_score:
        best_score = score
        best_val_psnr = val_psnr
        best_val_sipsnr = val_sipsnr

        torch.save({"model": model.state_dict()}, save_path)
        print(f"--> Best Fold {fold+1} model saved! ({score_name}: {best_score:.4f})")

  print(f"\n--- Fold {fold + 1} Training Complete. Executing FULL Re-Evaluation ---")

  best_checkpoint = torch.load(save_path, map_location=device)
  model.load_state_dict(best_checkpoint["model"])
  model.eval()


  final_fold_metrics = evaluate_model_25d(
      model=model, dataloader=val_loader_full, device=device,
      output_mode="clean", tta_mode="x8", data_range=1.0, clamp=True, pad_multiple=16, use_sipsnr=True, use_amp=False
  )

  final_val_psnr = final_fold_metrics["model_psnr"]
  final_val_sipsnr = final_fold_metrics.get("model_sipsnr", None)

  fold_best_psnrs.append(final_val_psnr)
  if final_val_sipsnr is not None:
    fold_best_sipsnrs.append(final_val_sipsnr)

  print(f"Fold {fold + 1} Official PSNR: {final_val_psnr:.4f} dB | SI-PSNR: {final_val_sipsnr:.4f} dB")

  if fold == 0:
    print(f"\nGenerating qualitative visual triplets for Fold 1...")
    visualize_25d_embryos(
        model=model, dataloader=val_loader_full, device=device,
        num_samples=3, output_mode="clean", tta_mode="x8", clamp=True, pad_multiple=16, cmap="viridis", show_error=True
    )

print("\n" + "=" * 60)
print("FINAL 2.5D 3-FOLD CROSS-VALIDATION SUMMARY")
print("=" * 60)
print(f"Mean PSNR:    {np.mean(fold_best_psnrs):.4f} ± {np.std(fold_best_psnrs):.4f} dB")
if fold_best_sipsnrs:
    print(f"Mean SI-PSNR: {np.mean(fold_best_sipsnrs):.4f} ± {np.std(fold_best_sipsnrs):.4f} dB")
print("=" * 60)